# eFOV final model: base model, then every parameter experiment

One notebook, run top to bottom (plan `claude/final_model_plan_2026-11-03.md`; decisions FM-01 to FM-33 in
`docs/decision_log.md`).

* **Part 1 - base model.** The reference slab: conditioning, beta, the two openCARP runs (intact and
  lesion), physiology checks, measurement windows, lead fields, electrograms, measurements, Wang's change
  points, the ground truth, the report and its figures.
* **Part 2 - experiments.** Each sweep variable takes each planned value alone (one at a time), plus any named
  variant of the sweep file. Each section runs its variants and compares them with the base model.
* **Part 3 - summary.** The pre-registered primary test (change-point shift per doubling of noise, FM-25) and
  one table of every variant.

Every stage records the settings it used; a rerun reuses completed stages and refuses stale ones, so the
notebook can be stopped and restarted. A failed variant is recorded with its error and the notebook carries
on; failures are never pooled. The same runs from a terminal: `python -m efov sweep config/sweeps/oat_baseline.json
--out runs/production --np 24`. Running the whole notebook takes days on the workstation; for an unattended
run, execute it with `jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1
notebooks/run_all.ipynb --output run_all_executed.ipynb` under `nohup`.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_SWEEP / EFOV_OUT / EFOV_NP ----
SWEEP = os.environ.get("EFOV_SWEEP", str(REPOSITORY / "config" / "sweeps" / "oat_baseline.json"))
OUT   = os.environ.get("EFOV_OUT",   str(REPOSITORY / "runs" / "production"))
NP    = int(os.environ.get("EFOV_NP", "24"))     # openCARP MPI processes and lead-field worker processes
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.sweep import Sweep, experimentFigure, experimentTable, noiseDoublingTable

pd.set_option("display.max_columns", 60, "display.width", 250)
sweep = Sweep(SWEEP, OUT, processCount=NP)
run = sweep.baseline                   # the base model = the sweep's baseline
config = run.config
FIGURES = Path(OUT) / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print(f"model {config['modelName']} ({sweep.modelConfigPath.name}); sweep {Path(SWEEP).name}; output {sweep.out}")
print(f"{len(sweep.plan) - 1} variants planned besides the baseline")
run.status()

# Part 1. The base model

## 1.1 What defines it
`config/model.par` holds every openCARP setting; `config/model.json` the sweep variables (baseline and
planned values) and settings openCARP cannot read (FM-01, FM-02). Nothing is simulated in this section.

In [ ]:
print(open(config["modelPar"]).read())

In [ ]:
pd.DataFrame({k: dict(baseline=v["baseline"], planned=v["planned"], actsOn=v.get("actsOn", ""))
              for k, v in config["sweep"].items()}).T

In [ ]:
import subprocess
print(subprocess.run([sys.executable, "-m", "efov", "plan", "--config", str(sweep.modelConfigPath)], cwd=REPOSITORY,
                     capture_output=True, text=True).stdout)

## 1.2 Simulate: conditioning, beta and the two openCARP runs
* **preflight**: software versions, repository commit, solver flavour, R availability.
* **conditioning**: one cell paced 1,000 times with `bench` (limit cycle).
* **tune**: beta for the CV target along fibres at the production mesh (FM-18, FM-32); taken from
  `config/tuning.json` when it holds this mesh and target.
* **simulation**: `efov/opencarp/slab_experiment.py` (carputils), intact and lesion slab, monodomain (FM-23),
  one resolved parameter file per run; in the lesion run a 5.6 mm ablation strip separates the paced tissue
  from isolated viable tissue (FM-24).

In [ ]:
run.runStage("preflight")

In [ ]:
run.runStage("conditioning")

In [ ]:
run.runStage("tune")

In [ ]:
result = run.runStage("simulation")
pd.DataFrame(result["timing"]).T.join(pd.DataFrame(result["nodes"]).T)

The resolved parameter file of the lesion run: model.par with this run's overrides in place, then the run paths.

In [ ]:
print((run.jobs["lesion"] / "run.par").read_text()[-3000:])

## 1.3 Physiology checks and measurement windows
Catheter-facing tissue nodes, last three beats: one activation per beat, repolarisation inside the response
window, quiet lesion core, planar wave in the intended direction, stable latency, amplitude and APD90.
Engineering criteria, not validated biological limits. The slab surface CV is reported, not gated: with bilayer
fibres it is below the 0.90 m/s along-fibre target (about 0.75 m/s, FM-33). Each placement's window comes from
the intact activation under it, before any electrogram is read (DL-2026-10-28-08, FM-28).

In [ ]:
physiology = run.runStage("physiology")
display(pd.DataFrame({c: v["criteria"] for c, v in physiology["conditions"].items()}))
{c: round(v["lastBeatSurfaceSpeedMPerS"], 3) for c, v in physiology["conditions"].items()}

In [ ]:
pd.read_csv(run.shared / "physiology" / "physiologyChecks.csv")

In [ ]:
window = run.runStage("window")
{k: v for k, v in window.items() if k != "windowsMs"}

## 1.4 Lead fields and electrograms
Each electrode is the set of bath mesh nodes inside its 1 mm cylinder (FM-14). One openCARP run per condition
(openCARP >= 20, `config/leadfield.par`, FM-35) injects 1 uA over each electrode (spread by volume), with the corner
grounded, and solves for its lead field Z (operator sigma_i + sigma_e in tissue, g_bath in the bath). It then streams
the monodomain Vm and writes every electrode's unipolar signal V(t) = Z . (K_i Vm(t)), the extracellular potential
with the sigma_i source (FM-23). The electrograms stage takes the analysis frames and checks openCARP's signals
against -Z^T S_i Vm formed with the tensors the ground truth uses. Lead fields are cached per conductor and reused
by later experiments.

In [ ]:
run.runStage("leadFields")

In [ ]:
run.runStage("electrograms")
egm = np.load(run.out / "electrograms" / "intact.npz")
print({k: egm[k].shape for k in egm.files})

## 1.5 Measurements, export and Wang's change points
Acquisition chain (FM-09, FM-22): resample to 2 kHz, Gaussian noise per electrode, 30-500 Hz band-pass for
bipoles and omnipoles. Peak frequency: two co-primary definitions (FM-10, FM-27); bipolar/omnipolar PP within
+-30 ms of the peak-frequency time (FM-11). Amplitudes stay in model units; Wang's 1.44 mV is shown beside them,
never applied (FM-25). Wang's Rmd runs unchanged except six one-slab functions (R/wang_replication.R, FM-12).

In [ ]:
measures = run.runStage("measurements")
display(pd.DataFrame(measures["amplitudes"]["modelFarViable"]))
measures["checks"]

In [ ]:
run.runStage("export")
changes = run.runStage("changePoints")
print("eligible for interpretation:", changes["eligibleForInterpretation"], changes["prerequisites"])
pd.read_csv(run.out / "changePoints" / "SuppResults5_change_points_all.csv")

## 1.6 Ground truth: the contributing tissue
For every recording at the ground-truth placements, each tissue element's signed share of the signal energy,
from the same lead fields as the electrograms. Shares sum to 1; cancellation is kept. Classes: near/far
(4 mm convention, FM-30) x endocardial/epicardial half (FM-15); transmural and lateral profiles; 3D fields
(`groundTruth/fields_<condition>.npz`, `groundTruth/shares_<condition>_gap<g>mm.vtk` for ParaView).

In [ ]:
checks = run.runStage("groundTruth")
summary = pd.read_csv(run.out / "groundTruth" / "summary.csv")
classes = [c for c in summary.columns if c.startswith("mass_")]
display(summary.groupby(["condition", "kind", "gapMm"])[classes].median())
checks

## 1.7 Report, acceptance and the base-model figures
`acceptance.json` lists every check; **modelAccepted** requires tuning, the ground-truth share sum, physiology
and the profile checks to pass. Change points are interpreted only if `eligibleForInterpretation` is true.

In [ ]:
report = run.runStage("report")
run.runStage("compact")                 # the baseline is never compacted (it is shared by later experiments)
sweep.runBaseline()                     # records the baseline as complete (every stage is reused)
display(Markdown((run.out / "report.md").read_text()))

In [ ]:
for path in report["figures"]:
    print(Path(path).name)
    display(Image(path))

# Part 2. Parameter experiments

Each section runs one experiment's variants, then shows them beside the base model: a table of headline
outcomes and a figure (`<OUT>/figures/experiment_<variable>.png`). Headline outcomes (definitions in
`efov.sweep.variantSummary`): acceptance and eligibility flags; Wang's voltage and peak-frequency change points
on the clean data and their median over the noisy realisations (posterior mean, mm from the scar boundary);
median far-field PP (model units); and, from the
ground truth, the median near-field and epicardial |c|-mass fractions. Every variant has its full run folder
(`<OUT>/variants/<name>/`: report.md, figures, tables) for detail.

Variants that change only electrodes, noise or analysis reuse the base-model simulation; lead fields are shared
by every variant with the same conductor (FM-23); variants with their own simulation are compacted after their
report (FM-20). Change points in a variant whose `eligibleForInterpretation` is false are not interpreted.

### Experiment: `wallMm`

Baseline `1.9`; values `1.0`, `1.5`, `2.0`, `2.5`, `3.0`; acts on *geometry*.

FM-48 (Matilda, 12 Nov 2026): mean left atrial wall thickness 1.89 +- 0.48 mm over 12 sites in 60 patients (Beinart et al. 2011, J Cardiovasc Electrophysiol 22:1232-1236, CT; value as given in the Methodology, primary paper not re-read here [VERIFY]), rounded to the 0.1 mm transmural mesh. Replaces the inferior-wall value 1.1 mm of FM-08 (Whitaker et al. 2021 median 1.14 mm), which the planned 1.0 mm variant now brackets. Must be a multiple of fixed.tissue.spacingTransmuralMm.

In [ ]:
VARIABLE = 'wallMm'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `fibres`

Baseline `{"model": "bilayer", "endocardialAngleDeg": 0.0, "epicardialAngleDeg": 90.0, "transitionDepthFraction": 0.5, "transitionWidthMm": 0.0}`; values `{"model": "uniform", "angleDeg": 0.0}`, `{"model": "linearRotation", "endocardialAngleDeg": 0.0, "epicardialAngleDeg": 90.0}`; acts on *geometry (.lon written per run)*.

Pashakhanloo et al. 2016: near-perpendicular layers, abrupt mid-wall transition in the posterior-inferior LA; constant on the roof. Angles are assumptions (none reported numerically). Angle from +x, counter-clockwise seen from the catheter (FM-06).

In [ ]:
VARIABLE = 'fibres'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `epicardialMedium`

Baseline `"none"`; values `"blood"`; acts on *geometry*.

none = no-flux epicardium (bath on the catheter side and laterally); blood = 5 mm bath on every side as Leenknegt et al. 2023 (FM-05).

In [ ]:
VARIABLE = 'epicardialMedium'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `scar`

Baseline `"ablation"`; values `"ablationPorated"`, `"insulating"`; acts on *model.par overrides*.

FM-07. ablation = healthy extracellular tensor (model.par); ablationPorated = sigma_i + sigma_e of healthy tissue (theoretical upper bound); insulating = earlier reference case. Intracellular lesion conductivity 1e-6 S/m in all three.

In [ ]:
VARIABLE = 'scar'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `waveAngleDeg`

Baseline `0.0`; values `30.0`, `60.0`, `90.0`; acts on *geometry (stimulus band)*.

Angle of the wave direction from +x; 0 = parallel to the lesion boundary, 90 = travelling toward the lesion (FM-17). Wang mapped during distal CS pacing: wavefronts reach the line at varied angles.

In [ ]:
VARIABLE = 'waveAngleDeg'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `gapsMm`

Baseline `[0.25, 1.0]`; values `[0.5]`, `[2.0]`; acts on *electrodes (analysis)*.

Electrode surface to tissue distance. Baseline 0.25 and 1 mm = the two contact cases of Rodrigo et al. 2026 (PLoS Comput Biol 22:e1014493; cylindrical electrodes 0.5 mm radius, 1 mm long). 3 mm removed (FM-40): its electrode top would lie 1 mm below the no-flux top of the 5 mm bath.

In [ ]:
VARIABLE = 'gapsMm'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `electrodeRadiusMm`

Baseline `0.5`; values `0.25`, `1.0`; acts on *electrodes (analysis)*.

HD Grid 1 mm electrodes (Wang supplement); Methodology diameter sweep 0.5-2 mm.

In [ ]:
VARIABLE = 'electrodeRadiusMm'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `bipoleOrientation`

Baseline `"selected"`; values `"x"`, `"y"`; acts on *measurements (analysis)*.

FM-39. selected = the larger peak-to-peak of the two orthogonal 4 mm pairs of each grid square, as EnSite does with the HD Grid (Siddiqui 2019, EP Lab Digest 19(2)); x = the pair along x (parallel to the lesion boundary and to the baseline wave); y = the pair along y (across the boundary). Orientation changes bipolar amplitude (Hwang et al. 2019, PLoS Comput Biol 15:e1006765).

In [ ]:
VARIABLE = 'bipoleOrientation'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `noiseSdMv`

Baseline `0.005`; values `0.0025`, `0.01`, `0.02`, `0.04`; acts on *measurements (analysis)*.

Gaussian white noise per electrode at the acquisition rate, in model mV (no calibration, FM-25). Assumption [VERIFY]: no source. The planned values form a doubling series; the pre-registered primary test is the change-point shift per doubling of noise (FM-25). FM-44: 0.04 mV added so the series reaches higher relative noise floors (noise peak-to-peak / healthy bipolar median, measurements/amplitudes.json): the model's amplitudes are not calibrated, so the level that matches a clinical signal-to-noise ratio is not known in advance and may lie above 0.02 mV.

In [ ]:
VARIABLE = 'noiseSdMv'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

### Experiment: `cvTargetMPerS`

Baseline `0.9`; values `0.7`, `1.25`; acts on *tune stage (beta)*.

van Schie et al. 2021 LA median 0.90 m/s; 0.70 and 1.25 m/s bracket reported central values (DL-2026-10-28-03).

In [ ]:
VARIABLE = 'cvTargetMPerS'
names = sweep.namesFor(VARIABLE)
if not names:
    print(f"{VARIABLE} is not in {Path(SWEEP).name}: skipped")
else:
    for name in names:
        print(f"--- {name}: overrides {sweep.plan[name]}", flush=True)
        entry = sweep.runVariant(name)
        print(f"    {entry['status']}" + (f": {entry['error']}" if entry["status"] != "complete" else ""), flush=True)
    display(experimentTable(sweep, VARIABLE))
    figure = experimentFigure(sweep, VARIABLE, FIGURES / f"experiment_{VARIABLE}.png")
    if figure:
        display(Image(str(figure)))

# Part 3. Summary

## 3.1 Pre-registered primary test: change-point shift per doubling of noise (FM-25)
For each of Wang's change points: the median over noisy realisations at each noise level, and the shift from
the previous (half) level. A change point that moves with the noise floor measures detectability, not
sensing volume. Criterion and interpretation as pre-registered in FM-25.

In [ ]:
doubling = noiseDoublingTable(sweep.out, sweep.modelConfigPath)
doubling if doubling is not None else "No noise variants have change points yet"

## 3.2 Every variant
Status of every variant (failed ones with their error) and the long table of headline outcomes, saved as
`<OUT>/sweepSummary.csv`.

In [ ]:
status = pd.DataFrame({k: dict(status=v["status"], overrides=json.dumps(v.get("overrides")), error=v.get("error"))
                       for k, v in sweep.status.items()}).T
display(status)
table = sweep.summary()
table.to_csv(sweep.out / "sweepSummary.csv", index=False)
table